<a href="https://colab.research.google.com/github/yuvikaagarwal21/EEG-PBL/blob/main/EEG_PROJECT_PHYSIONET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mne wfdb

In [ ]:
!wget -r -N -c -np https://physionet.org/files/eegmat/1.0.0/

In [ ]:
import mne

file_path = "physionet.org/files/eegmat/1.0.0/Subject01_1.edf"
raw = mne.io.read_raw_edf(file_path, preload=True)

raw

In [ ]:
raw.plot(n_channels=10, duration=5)

In [ ]:

raw_filtered = raw.copy().filter(l_freq=1, h_freq=45)


raw_filtered = raw_filtered.notch_filter(freqs=50)
raw_filtered._data = (raw_filtered._data - raw_filtered._data.mean(axis=1, keepdims=True)) / raw_filtered._data.std(axis=1, keepdims=True)

print("Bandpass + Notch filtering done")

raw_filtered.plot(scalings=dict(eeg=5), duration=5)

In [ ]:
import numpy as np

# Get filtered data
data = raw_filtered.get_data()

sfreq = raw_filtered.info['sfreq']  # sampling frequency
window_size = int(2 * sfreq)  # 2-second window

segments = []

for start in range(0, data.shape[1] - window_size, window_size):
    segment = data[:, start:start+window_size]
    segments.append(segment)

segments = np.array(segments)

print("Segments shape:", segments.shape)

In [ ]:
labels = np.zeros(len(segments))  # 0 = rest

print("Labels shape:", labels.shape)

In [ ]:
import mne

raw2 = mne.io.read_raw_edf(
    "physionet.org/files/eegmat/1.0.0/Subject01_2.edf",
    preload=True
)

raw2_filtered = raw2.filter(1., 45.)
raw2_filtered = raw2_filtered.notch_filter(freqs=50)
raw2_filtered._data = (raw2_filtered._data - raw2_filtered._data.mean(axis=1, keepdims=True)) / raw2_filtered._data.std(axis=1, keepdims=True)

In [ ]:
data2 = raw2_filtered.get_data()

sfreq2 = raw2_filtered.info['sfreq']
window_size2 = int(2 * sfreq2)

segments2 = []

for start in range(0, data2.shape[1] - window_size2, window_size2):
    segment = data2[:, start:start+window_size2]
    segments2.append(segment)

segments2 = np.array(segments2)

print("Arithmetic segments shape:", segments2.shape)

In [ ]:
labels2 = np.ones(len(segments2))  # 1 = arithmetic

print("Arithmetic labels shape:", labels2.shape)

In [ ]:
X = np.concatenate((segments, segments2), axis=0)
y = np.concatenate((labels, labels2), axis=0)

print("Final data shape:", X.shape)
print("Final labels shape:", y.shape)